# 02 — Unsupervised Clustering

KMeans (k=16, matching the known class count for now — a future pass
should sweep k and pick the value maximizing clustering quality rather
than assuming 16 is optimal) and HDBSCAN over each cached **summary**
embedding (UMAP-reduced first), scored against the hidden 16-way
ground-truth labels via Hungarian-matched accuracy and standard
clustering metrics. Silhouette/Davies-Bouldin are sub-sampled
(`config.SILHOUETTE_SAMPLE_SIZE`) since they're O(n^2). Each method also
saves a full-row output CSV (`text, summary, predicted_label, true_label`
for every row), not just the qualitative cluster-inspection sample.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import json

import hdbscan
import pandas as pd
import umap
from sklearn.cluster import KMeans

from utils import config
from utils.data import stratified_sample
from utils.embeddings import load_cached
from utils.interpretability import summarize_clusters
from utils.metrics import evaluate_unsupervised, hungarian_match_predictions
from utils.samples import save_full_output

In [2]:
train_clean = pd.read_parquet(config.PROCESSED_DIR / "train_clean.parquet")

train_sample = stratified_sample(train_clean, config.SAMPLE_SIZE, seed=config.SEED)
suffix = f"n{config.SAMPLE_SIZE}" if config.SAMPLE_SIZE else "full"
true_labels = train_sample["label"].to_numpy()
texts = train_sample["text"].tolist()
summaries = train_sample["summary"].tolist()

METHODS = ["tfidf", "minilm", "roberta"]
if config.OPENAI_API_KEY and load_cached(f"openai_train_{suffix}_summary") is not None:
    METHODS.append("openai")

embeddings_by_method = {}
for method in METHODS:
    arr = load_cached(f"{method}_train_{suffix}_summary")
    assert arr is not None, f"Missing cached embeddings for '{method}' — run 01_embeddings.ipynb first"
    embeddings_by_method[method] = arr

print(f"Clustering {len(train_sample)} rows across {len(embeddings_by_method)} embedding methods, k={config.NUM_CLASSES}")

Clustering 319 rows across 3 embedding methods, k=16


In [3]:
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = {}
runs = {}  # name -> (cluster_labels, emb_reduced) for the interpretability step below

for method, emb in embeddings_by_method.items():
    reducer = umap.UMAP(n_components=50, metric="cosine", random_state=config.SEED)
    emb_reduced = reducer.fit_transform(emb)

    kmeans = KMeans(n_clusters=config.NUM_CLASSES, random_state=config.SEED, n_init=10)
    km_labels = kmeans.fit_predict(emb_reduced)
    km_metrics = evaluate_unsupervised(true_labels, km_labels, emb_reduced,
                                        metric_sample_size=config.SILHOUETTE_SAMPLE_SIZE, seed=config.SEED)
    all_results[f"{method}_kmeans"] = km_metrics
    runs[f"{method}_kmeans"] = (km_labels, emb_reduced)

    clusterer = hdbscan.HDBSCAN(min_cluster_size=50, metric="euclidean", cluster_selection_method="eom")
    hdb_labels = clusterer.fit_predict(emb_reduced)
    hdb_metrics = evaluate_unsupervised(true_labels, hdb_labels, emb_reduced,
                                         metric_sample_size=config.SILHOUETTE_SAMPLE_SIZE, seed=config.SEED)
    all_results[f"{method}_hdbscan"] = hdb_metrics
    runs[f"{method}_hdbscan"] = (hdb_labels, emb_reduced)

    print(f"{method}: KMeans ACC={km_metrics['ACC (Hungarian)']:.3f} | "
          f"HDBSCAN coverage={hdb_metrics['Coverage']:.2f} ACC={hdb_metrics['ACC (Hungarian)']:.3f}")

for name, metrics in all_results.items():
    with open(config.RESULTS_DIR / f"metrics_{name}.json", "w") as f:
        json.dump(metrics, f, indent=2)

print(f"Saved {len(all_results)} result files to {config.RESULTS_DIR}")

C:\Users\ACER\OneDrive\Documents\final-project\.claude\worktrees\master-data-pipeline\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


tfidf: KMeans ACC=0.257 | HDBSCAN coverage=0.00 ACC=0.000


C:\Users\ACER\OneDrive\Documents\final-project\.claude\worktrees\master-data-pipeline\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


minilm: KMeans ACC=0.480 | HDBSCAN coverage=0.00 ACC=0.000


C:\Users\ACER\OneDrive\Documents\final-project\.claude\worktrees\master-data-pipeline\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


roberta: KMeans ACC=0.320 | HDBSCAN coverage=0.00 ACC=0.000
Saved 6 result files to C:\Users\ACER\OneDrive\Documents\final-project\.claude\worktrees\master-data-pipeline\results


### Cluster inspection + full-row output

The metrics above are permutation-invariant number-matching between
cluster IDs and true label IDs — they don't show *what a cluster is
about*. This prints, per method+algorithm, each cluster's size, majority
true label, purity, top KeyBERT key-phrases (from the summaries), and
example summaries nearest its centroid, then saves both the qualitative
sample and a full-row `text, summary, predicted_label, true_label` CSV
for every document (predicted label = the cluster's Hungarian-matched
class name).

In [4]:
for name, (cluster_labels, emb_reduced) in runs.items():
    summary = summarize_clusters(summaries, cluster_labels, true_labels, emb_reduced, config.CLASS_NAMES)

    print(f"\n=== {name} ===")
    if summary.empty:
        print("(no non-noise clusters — everything was noise)")
    else:
        with pd.option_context("display.max_colwidth", 60):
            print(summary.drop(columns="example_docs").to_string(index=False))
        for _, row in summary.iterrows():
            print(f"  cluster {row['cluster']} examples:")
            for doc in row["example_docs"]:
                print(f"    - {doc}")
        summary.to_csv(config.RESULTS_DIR / f"clusters_{name}.csv", index=False)

    predicted = hungarian_match_predictions(true_labels, cluster_labels)
    save_full_output(
        texts, predicted, true_labels, config.CLASS_NAMES,
        extra_columns={"summary": summaries},
        path=config.RESULTS_DIR / f"full_labels_{name}.csv")

print(f"\nSaved per-cluster qualitative summaries and full-row outputs to {config.RESULTS_DIR}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


=== tfidf_kmeans ===
 cluster  size majority_true_label   purity                                                                                                                                                                                                                                                           top_terms
       0    14           EDUCATION 0.214286                   shut canberra office, opic support renewable, emergencies shut canberra, holocaust education expanded, orphaned baby bats, nadal sensationally lost, titles roger federer, rafael nadal sensationally, number rafael nadal, novak djokovic rafael
       1    22       ENTERTAINMENT 0.318182                                         ben affleck looks, memorable news bloopers, darren criss grieving, explosion fedex facility, hackers breach, opera mozart operas, brother charles criss, news bloopers recent, harlow skin condition, color blotches harlow
       2    17               CRIME 0.294118                       

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


=== tfidf_hdbscan ===
(no non-noise clusters — everything was noise)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

...


=== minilm_hdbscan ===
(no non-noise clusters — everything was noise)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


=== roberta_kmeans ===
 cluster  size majority_true_label   purity                                                                                                                                                                                                                                                       top_terms
       0    15               MEDIA 0.333333                                                  wears vr headset, vr headset isn, steele trump evangelical, lemon unloads kanye, fired political director, news anchor sinclair, trump genius graham, cbs news fired, trump praised sinclair, headset isn real
       1    26              SPORTS 0.538462                                                   memorable news bloopers, world, di matteo sacked, rafa nadal crashes, nadal crashes wimbledon, bolshoi theatre killed, united moved second, google shares fall, shut canberra office, manchester united moved
       2    21           EDUCATION 0.285714 profit colleges defined, teacher

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


=== roberta_hdbscan ===
(no non-noise clusters — everything was noise)

Saved per-cluster qualitative summaries and full-row outputs to C:\Users\ACER\OneDrive\Documents\final-project\.claude\worktrees\master-data-pipeline\results
